# Trees

*Represent a category hierarchy, identify parent-child relationships, and reconstruct paths from the root.*

Hierarchical data records which items contain or belong under other items. In a category hierarchy, each **node** is a category and each **edge** connects two related categories. Keeping only the category names would lose those relationships.

A **tree** is a connected undirected graph without cycles. Choosing a root organizes the tree into parent-child relationships. We represent one category tree with Python dictionaries and lists, then reuse it to find a parent and trace a root-to-leaf path.


## Parent-Child Relationships

In a rooted tree, every node except the **root** has exactly one parent. A node may have several children. A **leaf** has no children; a **descendant** is reached by following one or more child links.

| Term | Meaning in the category tree |
| --- | --- |
| Root | `data`, the starting point of the hierarchy |
| Parent and child | `numeric` is the parent of `discrete` |
| Siblings | `discrete` and `continuous` share the same parent |
| Leaf | `ordinal` has no children |
| Descendant | `ordinal` is a descendant of `data`, through `categorical` |

The numeric categories **discrete** and **continuous** describe measurements, such as event counts and measured duration, rather than Python storage types. The arrows in the illustration point from parent to child.

A dictionary maps each node name to its child list. Empty lists retain the leaves as nodes. The comprehension selects nodes whose child lists are empty; summing child counts counts each parent-child edge once.


**Predict before viewing the figure or running the code.** The root has two children, and each of those has two leaf children. How many nodes and edges does the tree have?


<img src="https://raw.githubusercontent.com/sonamu-jun/introduction-to-bigdata/main/02-2_Data_Structures/assets/07_trees/tree_hierarchy.webp" width="480" alt="The root data has children numeric and categorical. Numeric has leaves discrete and continuous; categorical has leaves nominal and ordinal. The path data to categorical to ordinal is highlighted.">

Amber follows the two-edge path from root to ordinal.


In [1]:
children_by_node = {
    "data": ["numeric", "categorical"],
    "numeric": ["discrete", "continuous"],
    "categorical": ["nominal", "ordinal"],
    "discrete": [],
    "continuous": [],
    "nominal": [],
    "ordinal": [],
}

leaf_nodes = [node for node, children in children_by_node.items() if not children]
tree_edge_count = sum(len(children) for children in children_by_node.values())

print("Root children:", children_by_node["data"])
print("Numeric children:", children_by_node["numeric"])
print("Leaves:", sorted(leaf_nodes))
print("Node count:", len(children_by_node))
print("Parent-child edge count:", tree_edge_count)


Root children: ['numeric', 'categorical']
Numeric children: ['discrete', 'continuous']
Leaves: ['continuous', 'discrete', 'nominal', 'ordinal']
Node count: 7
Parent-child edge count: 6


**Check:** The tree has seven nodes, six edges, and four leaves. `discrete` is a descendant of `data`, but it is not one of the root's immediate children.

A tree with `n` nodes has `n - 1` edges and exactly one path without repeated nodes between any two nodes. Adding a connection between the two branches would create a cycle. The dictionary stores the relationships; it does not enforce the tree's rules.


## Paths Through a Hierarchy

A **path** is a sequence of connected nodes. Its length counts edges, so a path containing three nodes has length two. A node's **depth** is the length of its path from the root; the root has depth zero.

The child lists support moving down the tree. To move upward, first build `parent_by_node`, a reverse lookup from each child to its parent. The nested loops read each parent-child pair once. Starting at `ordinal`, the `while` loop follows parents until it reaches the root, which has no entry in the parent lookup. `reversed()` puts the result in root-to-leaf order.

This code assumes the prepared tree is valid: each non-root node has one parent and there are no cycles. The highlighted route in the same illustration shows the path being reconstructed.


In [2]:
parent_by_node = {}
for parent, children in children_by_node.items():
    for child in children:
        parent_by_node[child] = parent

target_node = "ordinal"
node = target_node
path_to_root = [node]
while node in parent_by_node:
    node = parent_by_node[node]
    path_to_root.append(node)

root_to_leaf = list(reversed(path_to_root))
print("Parent of ordinal:", parent_by_node[target_node])
print("Root-to-leaf path:", " -> ".join(root_to_leaf))
print("Path length in edges:", len(root_to_leaf) - 1)
print("Ordinal depth:", len(root_to_leaf) - 1)


Parent of ordinal: categorical
Root-to-leaf path: data -> categorical -> ordinal
Path length in edges: 2
Ordinal depth: 2


The result is `data -> categorical -> ordinal`. `categorical` is the immediate parent, while `data` is an ancestor. The two edges place `ordinal` at depth two. A flat set of category names would retain membership but could not recover this route.


Use a rooted tree when each non-root item belongs to one parent in a hierarchy. Child lists support moving downward; a parent lookup supports moving upward and reconstructing paths. Keep the relationships as well as the node names, and distinguish a direct parent from a more distant ancestor.
